# Coffee quality benchmark – inference pipeline with MLflow

MLflow based version of
`coffee-quality-pricing-benchmark_inference_pipeline.ipynb`. The model
is loaded from the MLflow Model Registry (`coffee-quality-model`,
alias `champion`) created by
`coffee-quality-pricing-benchmark_training_pipeline_mlflow.ipynb`, and
every inference run is tracked in MLflow in the
`coffeePricePredictor/artifacts` folder.

**Steps**

1. Configuration and MLflow setup
2. Loading the registered model
3. Loading and validating the input data
4. Prediction
5. Logging the inference run (parameters, input dataset, prediction
   statistics and the predictions CSV as an MLflow artifact)
6. Optional evaluation (only if the input contains the target column)
7. Checking the logged artifacts

The input CSV has to contain the feature columns of the training data.
Identifier, target and other extra columns are ignored; `record_id` is
kept in the output when present.

In [1]:
"""Inference pipeline for the coffee quality model with MLflow."""

import os
import tempfile
from pathlib import Path

import mlflow
import mlflow.sklearn
import numpy as np
import pandas as pd
from mlflow import MlflowClient
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score,
)
from sklearn.pipeline import Pipeline

2026/10/21 16:01:09 INFO mlflow.agent.hint: Load the `instrumenting-with-mlflow-tracing` skill at C:\Users\bordasm\.venv\Lib\site-packages\mlflow\assistant\skills\instrumenting-with-mlflow-tracing\SKILL.md before writing any tracing code; it ships with this MLflow install. Set MLFLOW_DISABLE_AGENT_HINT=1 to silence this.


## 1. Configuration and MLflow setup

Paths are resolved relative to the `pipelines` folder. Set `INPUT_PATH`
to the CSV file to be scored.

In [2]:
PIPELINE_DIR = Path.cwd()
PROJECT_ROOT = PIPELINE_DIR.parent
INPUT_PATH = PROJECT_ROOT / "data" / "coffee_inference_sample.csv"
ARTIFACTS_DIR = PROJECT_ROOT / "artifacts"
TRACKING_URI = "sqlite:///" + (ARTIFACTS_DIR / "mlflow.db").as_posix()
MLRUNS_DIR = ARTIFACTS_DIR / "mlruns"

EXPERIMENT_NAME = "coffee-quality-inference"
REGISTERED_MODEL_NAME = "coffee-quality-model"
MODEL_ALIAS = "champion"

TARGET_COLUMN = "coffee_quality_score"
ID_COLUMN = "record_id"
PREDICTION_COLUMN = "predicted_coffee_quality_score"
PREDICTIONS_FILE_NAME = "coffee_quality_predictions.csv"

os.environ["MLFLOW_ENABLE_ARTIFACTS_PROGRESS_BAR"] = "false"

print(f"Input path:   {INPUT_PATH}")
print(f"Tracking URI: {TRACKING_URI}")

Input path:   c:\D\Work\Ismeret\AI_ML_DL_TL\CUBIX_MLOPS_AIOPS\coffeePricePredictor\data\coffee_inference_sample.csv
Tracking URI: sqlite:///c:/D/Work/Ismeret/AI_ML_DL_TL/CUBIX_MLOPS_AIOPS/coffeePricePredictor/artifacts/mlflow.db


In [3]:
def setup_mlflow(experiment_name: str) -> str:
    """Configure the tracking store and return the experiment id."""
    if not (ARTIFACTS_DIR / "mlflow.db").exists():
        raise FileNotFoundError(
            "MLflow store not found. Run the training pipeline first."
        )
    mlflow.set_tracking_uri(TRACKING_URI)

    experiment = mlflow.get_experiment_by_name(experiment_name)
    if experiment is None:
        experiment_id = mlflow.create_experiment(
            experiment_name,
            artifact_location=(MLRUNS_DIR / experiment_name).as_uri(),
        )
    else:
        experiment_id = experiment.experiment_id
    mlflow.set_experiment(experiment_name)
    return experiment_id


experiment_id = setup_mlflow(EXPERIMENT_NAME)
client = MlflowClient()

print(f"Experiment: {EXPERIMENT_NAME} (id: {experiment_id})")

Experiment: coffee-quality-inference (id: 2)


## 2. Loading the registered model

In [4]:
model_version = client.get_model_version_by_alias(
    REGISTERED_MODEL_NAME, MODEL_ALIAS
)
model_uri = f"models:/{REGISTERED_MODEL_NAME}@{MODEL_ALIAS}"
model: Pipeline = mlflow.sklearn.load_model(model_uri)
training_metrics = client.get_run(model_version.run_id).data.metrics

print(f"Model URI:     {model_uri}")
print(f"Version:       {model_version.version}")
print(f"Model:         {model_version.tags.get('model_name', 'n/a')}")
print(f"Training run:  {model_version.run_id}")
pd.Series(training_metrics).sort_index()

Model URI:     models:/coffee-quality-model@champion
Version:       1
Model:         LightGBM
Training run:  039950d61d654d569983d46b265eb6b2


fit_time_seconds    2.044000
test_mae            1.323586
test_r2             0.884598
test_rmse           1.670481
train_mae           1.173199
train_r2            0.909844
train_rmse          1.475656
dtype: float64

In [5]:
def get_column_groups(model: Pipeline) -> tuple[list[str], list[str]]:
    """Return the numeric and categorical columns of the preprocessor."""
    numerical_columns: list[str] = []
    categorical_columns: list[str] = []
    for name, _, columns in model.named_steps["preprocessor"].transformers_:
        if name == "num":
            numerical_columns = list(columns)
        elif name == "cat":
            categorical_columns = list(columns)
    return numerical_columns, categorical_columns


feature_columns = list(model.feature_names_in_)
numerical_columns, categorical_columns = get_column_groups(model)

print(f"Feature columns:     {len(feature_columns)}")
print(f"Numerical columns:   {len(numerical_columns)}")
print(f"Categorical columns: {len(categorical_columns)}")

Feature columns:     31
Numerical columns:   20
Categorical columns: 11


## 3. Loading and validating the input data

In [6]:
def load_input_data(input_path: Path) -> pd.DataFrame:
    """Load the CSV file to be scored."""
    if not input_path.exists():
        raise FileNotFoundError(f"Input file not found: {input_path}")

    dataframe = pd.read_csv(input_path)
    if dataframe.empty:
        raise ValueError(f"Input file contains no rows: {input_path}")
    return dataframe


def prepare_inference_features(
    dataframe: pd.DataFrame,
    feature_columns: list[str],
    numerical_columns: list[str],
    categorical_columns: list[str],
) -> pd.DataFrame:
    """Validate the input columns and cast them to the training types."""
    missing = [
        column for column in feature_columns if column not in dataframe
    ]
    if missing:
        raise ValueError(f"Input is missing feature columns: {missing}")

    features = dataframe[feature_columns].copy()
    for column in numerical_columns:
        try:
            features[column] = pd.to_numeric(features[column]).astype(
                "float64"
            )
        except (TypeError, ValueError) as error:
            raise ValueError(
                f"Column '{column}' must contain numeric values."
            ) from error
    for column in categorical_columns:
        series = features[column]
        features[column] = (
            series.astype(str).where(series.notna(), np.nan).astype(object)
        )
    return features

In [7]:
input_data = load_input_data(INPUT_PATH)
inference_features = prepare_inference_features(
    input_data,
    feature_columns,
    numerical_columns,
    categorical_columns,
)

print(f"Input shape: {input_data.shape}")
inference_features.head()

Input shape: (20, 32)


,harvest_season,harvest_calendar_year,harvest_month,country_of_origin,farm_type,farm_size_hectares,tree_age_years,altitude_mean_meters,species,avg_temp_c,...,traceability_level,certification,intended_buyer_segment,lot_size_kg,roast_level,acidity_score,sweetness_score,body_score,aroma_score,flavor_score
0,2021/22,2021.0,11.0,Colombia,Cooperative,15.60,21.0,2163.0,Arabica,12.5,...,Region,NaN,Premium,24372.0,Medium,8.17,7.88,7.17,7.84,8.15
1,2022/23,2023.0,5.0,Colombia,Smallholder,3.07,9.0,2079.0,Arabica,11.9,...,Country,NaN,Commodity,35448.0,Dark,7.42,7.55,7.86,8.57,7.75
2,2020/21,2020.0,12.0,Kenya,Cooperative,10.48,30.0,1973.0,Arabica,11.0,...,Cooperative,NaN,Premium,4810.0,Dark,8.33,7.54,8.94,8.77,7.86
3,2020/21,2020.0,12.0,Colombia,Cooperative,15.16,16.0,2003.0,Arabica,13.5,...,Country,Fair Trade,Commodity,25711.0,Dark,6.91,8.79,7.99,7.64,7.69
4,2022/23,2022.0,7.0,Indonesia,Cooperative,13.95,12.0,477.0,Robusta,23.3,...,Cooperative,Fair Trade,Commodity,16902.0,Medium,6.25,7.84,7.58,7.71,7.48


## 4. Prediction

In [8]:
def build_output(
    dataframe: pd.DataFrame,
    predictions: np.ndarray,
) -> pd.DataFrame:
    """Combine the record identifiers with the predictions."""
    if ID_COLUMN in dataframe.columns:
        output = dataframe[[ID_COLUMN]].copy()
    else:
        output = pd.DataFrame(index=dataframe.index)
    output[PREDICTION_COLUMN] = predictions
    return output


predictions = model.predict(inference_features)
prediction_output = build_output(input_data, predictions)

prediction_output.head(20)

,record_id,predicted_coffee_quality_score
0,REC-0012093,79.424517
1,REC-0022018,80.373375
2,REC-0019926,82.220487
3,REC-0027317,79.489755
4,REC-0011839,70.713240
5,REC-0008363,80.289786
6,REC-0004153,75.480498
7,REC-0018134,74.202564
8,REC-0026646,71.659487
9,REC-0032575,72.475490


## 5. Logging the inference run

The predictions CSV is stored as an MLflow artifact of the inference
run, together with the model version, the input dataset reference and
summary statistics of the predictions. If the input contains the true
target values, RMSE, MAE and R² are logged as well (step 6).

In [9]:
def compute_evaluation(
    dataframe: pd.DataFrame,
    predictions: np.ndarray,
) -> dict[str, float]:
    """Return evaluation metrics if the target column is available."""
    if TARGET_COLUMN not in dataframe.columns:
        return {}
    y_true = dataframe[TARGET_COLUMN]
    return {
        "eval_rmse": float(np.sqrt(mean_squared_error(y_true, predictions))),
        "eval_mae": float(mean_absolute_error(y_true, predictions)),
        "eval_r2": float(r2_score(y_true, predictions)),
    }


prediction_stats = {
    "prediction_mean": float(np.mean(predictions)),
    "prediction_std": float(np.std(predictions)),
    "prediction_min": float(np.min(predictions)),
    "prediction_max": float(np.max(predictions)),
}
evaluation = compute_evaluation(input_data, predictions)

if mlflow.active_run() is not None:
    mlflow.end_run(status="FAILED")

with mlflow.start_run(run_name="inference-pipeline") as inference_run:
    mlflow.set_tags(
        {
            "pipeline": "inference",
            "source": "inference_pipeline_mlflow.ipynb",
            "registered_model_name": REGISTERED_MODEL_NAME,
            "registered_model_version": model_version.version,
            "model_alias": MODEL_ALIAS,
        }
    )
    mlflow.log_params(
        {
            "model_uri": model_uri,
            "model_version": model_version.version,
            "input_path": str(INPUT_PATH),
            "n_rows": len(input_data),
        }
    )
    mlflow.log_input(
        mlflow.data.from_pandas(
            input_data,
            source=str(INPUT_PATH),
            name=INPUT_PATH.stem,
        ),
        context="inference",
    )
    mlflow.log_metrics({**prediction_stats, **evaluation})
    with tempfile.TemporaryDirectory() as temp_dir:
        predictions_path = Path(temp_dir) / PREDICTIONS_FILE_NAME
        prediction_output.to_csv(predictions_path, index=False)
        mlflow.log_artifact(str(predictions_path))

print(f"Inference run id: {inference_run.info.run_id}")
pd.Series(prediction_stats)

Inference run id: 4bd6fd985abd4519925a54c1a913f5ae


C:\Users\bordasm\.venv\Lib\site-packages\mlflow\data\dataset_source_registry.py:148: UserWarning: The specified dataset source can be interpreted in multiple ways: LocalArtifactDatasetSource, LocalArtifactDatasetSource. MLflow will assume that this is a LocalArtifactDatasetSource source.
  return _dataset_source_registry.resolve(
C:\Users\bordasm\.venv\Lib\site-packages\mlflow\types\utils.py:440: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/mod

prediction_mean    76.795833
prediction_std      4.266119
prediction_min     69.080491
prediction_max     85.058600
dtype: float64

## 6. Optional evaluation

In [10]:
if evaluation:
    print(pd.Series(evaluation))
else:
    print(f"'{TARGET_COLUMN}' is not in the input data; evaluation skipped.")

'coffee_quality_score' is not in the input data; evaluation skipped.


## 7. Checking the logged artifacts

The predictions are read back from the MLflow artifact store.

In [11]:
artifact_uri = (
    f"runs:/{inference_run.info.run_id}/{PREDICTIONS_FILE_NAME}"
)
with tempfile.TemporaryDirectory() as temp_dir:
    downloaded_path = mlflow.artifacts.download_artifacts(
        artifact_uri, dst_path=temp_dir
    )
    logged_predictions = pd.read_csv(downloaded_path)

if len(logged_predictions) != len(prediction_output):
    raise AssertionError("The logged predictions are incomplete.")

print(f"Artifact URI: {artifact_uri}")
logged_predictions.head()

Artifact URI: runs:/4bd6fd985abd4519925a54c1a913f5ae/coffee_quality_predictions.csv


,record_id,predicted_coffee_quality_score
0,REC-0012093,79.424517
1,REC-0022018,80.373375
2,REC-0019926,82.220487
3,REC-0027317,79.489755
4,REC-0011839,70.713240
